<a href="https://colab.research.google.com/github/Pabloflrs190468/ACTIVIDADES/blob/main/Pia_esp32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#include "modelo.h"

#include <TensorFlowLite_ESP32.h>

#include "tensorflow/lite/micro/micro_mutable_op_resolver.h"
#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/schema/schema_generated.h"

#include <max6675.h>

// -----------------------------------
// MAX6675
// -----------------------------------

int thermoDO = 19;
int thermoCS = 5;
int thermoCLK = 18;

MAX6675 thermocouple(
  thermoCLK,
  thermoCS,
  thermoDO
);

// -----------------------------------
// MQ5
// -----------------------------------

#define MQ5_PIN 34

// -----------------------------------
// ALARMAS
// -----------------------------------

#define ALARMA_TEMP 26
#define ALARMA_GAS 27

// -----------------------------------
// TENSORFLOW LITE
// -----------------------------------

const tflite::Model* model = nullptr;

tflite::MicroMutableOpResolver<5> resolver;

constexpr int tensorArenaSize = 8 * 1024;
uint8_t tensorArena[tensorArenaSize];

tflite::MicroInterpreter* interpreter = nullptr;

TfLiteTensor* input = nullptr;
TfLiteTensor* output = nullptr;

// -----------------------------------
// SETUP
// -----------------------------------

void setup() {

  Serial.begin(115200);

  pinMode(ALARMA_TEMP, OUTPUT);
  pinMode(ALARMA_GAS, OUTPUT);

  digitalWrite(ALARMA_TEMP, LOW);
  digitalWrite(ALARMA_GAS, LOW);

  // -----------------------------------
  // OPERACIONES IA
  // -----------------------------------

  resolver.AddFullyConnected();
  resolver.AddRelu();
  resolver.AddLogistic();

  // -----------------------------------
  // CARGAR MODELO
  // -----------------------------------

  model = tflite::GetModel(modelo_tflite);

  interpreter = new tflite::MicroInterpreter(
    model,
    resolver,
    tensorArena,
    tensorArenaSize
  );

  interpreter->AllocateTensors();

  input = interpreter->input(0);
  output = interpreter->output(0);

  Serial.println("TensorFlow Lite iniciado");

  delay(1000);
}

// -----------------------------------
// LOOP
// -----------------------------------

void loop() {

  // -----------------------------------
  // LEER SENSORES
  // -----------------------------------

  float temperatura = thermocouple.readCelsius();

  int gas = analogRead(MQ5_PIN);

  // -----------------------------------
  // ENTRADAS IA
  // -----------------------------------

  input->data.f[0] = temperatura;
  input->data.f[1] = gas;

  // -----------------------------------
  // INFERENCIA IA
  // -----------------------------------

  interpreter->Invoke();

  float prediccion = output->data.f[0];

  // -----------------------------------
  // MONITOR SERIAL
  // -----------------------------------

  Serial.print("Temperatura: ");
  Serial.print(temperatura);

  Serial.print("  MQ5: ");
  Serial.print(gas);

  Serial.print("  IA: ");
  Serial.println(prediccion);

  // -----------------------------------
  // DECISION IA
  // -----------------------------------

  if (prediccion > 0.5) {

    digitalWrite(ALARMA_TEMP, HIGH);
    digitalWrite(ALARMA_GAS, HIGH);

    Serial.println("PELIGRO DETECTADO");

  } else {

    digitalWrite(ALARMA_TEMP, LOW);
    digitalWrite(ALARMA_GAS, LOW);

    Serial.println("NORMAL");
  }

  delay(500);
}